#Initialize Spark and load manufacturing datasets.

In [ ]:
#Initialize Spark and load manufacturing datasets.
pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder.appName("Predictive Maintenance analysis").getOrCreate()

In [ ]:
df = spark.read.csv(
    "/content/ai4i2020.csv",
    header=True,
    inferSchema=True
)

In [ ]:
df.show(10)

+---+----------+----+-------------------+-----------------------+----------------------+-----------+---------------+---------------+---+---+---+---+---+
|UDI|Product ID|Type|Air temperature [K]|Process temperature [K]|Rotational speed [rpm]|Torque [Nm]|Tool wear [min]|Machine failure|TWF|HDF|PWF|OSF|RNF|
+---+----------+----+-------------------+-----------------------+----------------------+-----------+---------------+---------------+---+---+---+---+---+
|  1|    M14860|   M|              298.1|                  308.6|                  1551|       42.8|              0|              0|  0|  0|  0|  0|  0|
|  2|    L47181|   L|              298.2|                  308.7|                  1408|       46.3|              3|              0|  0|  0|  0|  0|  0|
|  3|    L47182|   L|              298.1|                  308.5|                  1498|       49.4|              5|              0|  0|  0|  0|  0|  0|
|  4|    L47183|   L|              298.2|                  308.6|                 

In [ ]:
df.printSchema()

root
 |-- UDI: integer (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Type: string (nullable = true)
 |-- Air temperature [K]: double (nullable = true)
 |-- Process temperature [K]: double (nullable = true)
 |-- Rotational speed [rpm]: integer (nullable = true)
 |-- Torque [Nm]: double (nullable = true)
 |-- Tool wear [min]: integer (nullable = true)
 |-- Machine failure: integer (nullable = true)
 |-- TWF: integer (nullable = true)
 |-- HDF: integer (nullable = true)
 |-- PWF: integer (nullable = true)
 |-- OSF: integer (nullable = true)
 |-- RNF: integer (nullable = true)



In [ ]:
print("Rows :", df.count())
print("Columns :", len(df.columns))


Rows : 10000
Columns : 14


#RDD Implementation


In [ ]:
 #RDD Implementation
 #Convert DataFrame to RDD
rdd = df.rdd
print("RDD Created Successfully")

RDD Created Successfully


In [ ]:
#display the 5 records
rdd.take(5)

[Row(UDI=1, Product ID='M14860', Type='M', Air temperature [K]=298.1, Process temperature [K]=308.6, Rotational speed [rpm]=1551, Torque [Nm]=42.8, Tool wear [min]=0, Machine failure=0, TWF=0, HDF=0, PWF=0, OSF=0, RNF=0),
 Row(UDI=2, Product ID='L47181', Type='L', Air temperature [K]=298.2, Process temperature [K]=308.7, Rotational speed [rpm]=1408, Torque [Nm]=46.3, Tool wear [min]=3, Machine failure=0, TWF=0, HDF=0, PWF=0, OSF=0, RNF=0),
 Row(UDI=3, Product ID='L47182', Type='L', Air temperature [K]=298.1, Process temperature [K]=308.5, Rotational speed [rpm]=1498, Torque [Nm]=49.4, Tool wear [min]=5, Machine failure=0, TWF=0, HDF=0, PWF=0, OSF=0, RNF=0),
 Row(UDI=4, Product ID='L47183', Type='L', Air temperature [K]=298.2, Process temperature [K]=308.6, Rotational speed [rpm]=1433, Torque [Nm]=39.5, Tool wear [min]=7, Machine failure=0, TWF=0, HDF=0, PWF=0, OSF=0, RNF=0),
 Row(UDI=5, Product ID='L47184', Type='L', Air temperature [K]=298.2, Process temperature [K]=308.7, Rotational 

In [ ]:
#Count Total Records
print("Total Records :", rdd.count())

Total Records : 10000


In [ ]:
#Transformation- Select Machine Type
machine_type = rdd.map(lambda row: row["Type"])
machine_type.take(10)

['M', 'L', 'L', 'L', 'L', 'M', 'L', 'L', 'M', 'M']

In [ ]:
failed = rdd.filter(lambda row: row["Machine failure"] == 1)
failed.take(5)

[Row(UDI=51, Product ID='L47230', Type='L', Air temperature [K]=298.9, Process temperature [K]=309.1, Rotational speed [rpm]=2861, Torque [Nm]=4.6, Tool wear [min]=143, Machine failure=1, TWF=0, HDF=0, PWF=1, OSF=0, RNF=0),
 Row(UDI=70, Product ID='L47249', Type='L', Air temperature [K]=298.9, Process temperature [K]=309.0, Rotational speed [rpm]=1410, Torque [Nm]=65.7, Tool wear [min]=191, Machine failure=1, TWF=0, HDF=0, PWF=1, OSF=1, RNF=0),
 Row(UDI=78, Product ID='L47257', Type='L', Air temperature [K]=298.8, Process temperature [K]=308.9, Rotational speed [rpm]=1455, Torque [Nm]=41.3, Tool wear [min]=208, Machine failure=1, TWF=1, HDF=0, PWF=0, OSF=0, RNF=0),
 Row(UDI=161, Product ID='L47340', Type='L', Air temperature [K]=298.4, Process temperature [K]=308.2, Rotational speed [rpm]=1282, Torque [Nm]=60.7, Tool wear [min]=216, Machine failure=1, TWF=0, HDF=0, PWF=0, OSF=1, RNF=0),
 Row(UDI=162, Product ID='L47341', Type='L', Air temperature [K]=298.3, Process temperature [K]=308.

In [ ]:
print("Failed Machines :", failed.count())#count the fail machines

Failed Machines : 339


In [ ]:
torque = rdd.map(lambda row: row["Torque [Nm]"])
torque.take(10)

[42.8, 46.3, 49.4, 39.5, 40.0, 41.9, 42.4, 40.2, 28.6, 28.0]

In [ ]:
torque.mean()

39.98691000000008

#Key-Value Operations and Persistence


In [ ]:
#Key-Value Operations and Persistence
rdd1=df.rdd

In [ ]:
 #Key-Value Operation
type_rdd = rdd1.map(lambda row: (row["Type"], 1))
print("machine type:")
type_rdd.take(5)


machine type:


[('M', 1), ('L', 1), ('L', 1), ('L', 1), ('L', 1)]

In [ ]:
# Shuffle Operation
count_type = type_rdd.reduceByKey(lambda a, b: a + b)
print("Machine Count")
print(count_type.collect())

Machine Count
[('M', 2997), ('L', 6000), ('H', 1003)]


In [ ]:
#(C) Persistence
count_type.persist()


PythonRDD[44] at collect at /tmp/ipykernel_443/3970004349.py:4

In [ ]:
print(count_type.is_cached)

True


#Spark DataFrame Operations

In [ ]:
print(df.columns)

['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']


In [ ]:
#Select Specific Columns
df.select(
    "Type",
    "Torque [Nm]",
    "Machine failure"
).show(10)

+----+-----------+---------------+
|Type|Torque [Nm]|Machine failure|
+----+-----------+---------------+
|   M|       42.8|              0|
|   L|       46.3|              0|
|   L|       49.4|              0|
|   L|       39.5|              0|
|   L|       40.0|              0|
|   M|       41.9|              0|
|   L|       42.4|              0|
|   L|       40.2|              0|
|   M|       28.6|              0|
|   M|       28.0|              0|
+----+-----------+---------------+
only showing top 10 rows


In [ ]:
failed = df.filter(df["Machine failure"] == 1)
failed.show(10)

+---+----------+----+-------------------+-----------------------+----------------------+-----------+---------------+---------------+---+---+---+---+---+
|UDI|Product ID|Type|Air temperature [K]|Process temperature [K]|Rotational speed [rpm]|Torque [Nm]|Tool wear [min]|Machine failure|TWF|HDF|PWF|OSF|RNF|
+---+----------+----+-------------------+-----------------------+----------------------+-----------+---------------+---------------+---+---+---+---+---+
| 51|    L47230|   L|              298.9|                  309.1|                  2861|        4.6|            143|              1|  0|  0|  1|  0|  0|
| 70|    L47249|   L|              298.9|                  309.0|                  1410|       65.7|            191|              1|  0|  0|  1|  1|  0|
| 78|    L47257|   L|              298.8|                  308.9|                  1455|       41.3|            208|              1|  1|  0|  0|  0|  0|
|161|    L47340|   L|              298.4|                  308.2|                 

In [ ]:
print("Failed Machines:", failed.count())

Failed Machines: 339


In [ ]:
df.groupBy("Type").avg("Air temperature [K]").show()

+----+------------------------+
|Type|avg(Air temperature [K])|
+----+------------------------+
|   M|      300.02926259592965|
|   L|       300.0158333333335|
|   H|       299.8669990029907|
+----+------------------------+



In [ ]:
df.groupBy("Type").min("Torque [Nm]").show()

+----+----------------+
|Type|min(Torque [Nm])|
+----+----------------+
|   M|             9.7|
|   L|             3.8|
|   H|            12.8|
+----+----------------+



Exploratory Data Analysis (EDA) & Spark SQL

In [ ]:
df.createOrReplaceTempView("machines")

In [ ]:
spark.sql("""
SELECT Type,
COUNT(*) AS Total_Machines
FROM machines
GROUP BY Type
""").show()

+----+--------------+
|Type|Total_Machines|
+----+--------------+
|   M|          2997|
|   L|          6000|
|   H|          1003|
+----+--------------+



In [ ]:
spark.sql("""
SELECT `Machine failure`,
COUNT(*) AS Total
FROM machines
GROUP BY `Machine failure`
""").show()

+---------------+-----+
|Machine failure|Total|
+---------------+-----+
|              1|  339|
|              0| 9661|
+---------------+-----+



In [ ]:
spark.sql("""
SELECT Type,
SUM(`Machine failure`) AS Failed
FROM machines
GROUP BY Type
""").show()

+----+------+
|Type|Failed|
+----+------+
|   M|    83|
|   L|   235|
|   H|    21|
+----+------+



In [ ]:
spark.sql("""
SELECT Type,
AVG(`Torque [Nm]`) AS Avg_Torque
FROM machines
GROUP BY Type
""").show()

+----+------------------+
|Type|        Avg_Torque|
+----+------------------+
|   M|40.017250583917296|
|   L| 39.99659999999998|
|   H| 39.83828514456634|
+----+------------------+



In [ ]:
spark.sql("""
SELECT Type,
AVG(`Air temperature [K]`) AS Avg_Temp
FROM machines
GROUP BY Type
""").show()

+----+------------------+
|Type|          Avg_Temp|
+----+------------------+
|   M|300.02926259592965|
|   L| 300.0158333333335|
|   H| 299.8669990029907|
+----+------------------+



In [ ]:
spark.sql("""
SELECT
SUM(CASE WHEN `Machine failure`=0 THEN 1 ELSE 0 END) AS Healthy,
SUM(CASE WHEN `Machine failure`=1 THEN 1 ELSE 0 END) AS Failed
FROM machines
""").show()

+-------+------+
|Healthy|Failed|
+-------+------+
|   9661|   339|
+-------+------+



In [ ]:
total = df.count()
failed = df.filter(df["Machine failure"] == 1).count()
print("Total Machines:", total)
print("Failed Machines:", failed)
print("Production Efficiency:", ((total - failed) / total) * 100)

Total Machines: 10000
Failed Machines: 339
Production Efficiency: 96.61


#
ETL Pipeline

In [ ]:
#Extract
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder.appName("ETL Pipeline").getOrCreate()

In [ ]:
df = spark.read.csv(
    "/content/ai4i2020.csv",
    header=True,
    inferSchema=True
)
df.show(5)

+---+----------+----+-------------------+-----------------------+----------------------+-----------+---------------+---------------+---+---+---+---+---+
|UDI|Product ID|Type|Air temperature [K]|Process temperature [K]|Rotational speed [rpm]|Torque [Nm]|Tool wear [min]|Machine failure|TWF|HDF|PWF|OSF|RNF|
+---+----------+----+-------------------+-----------------------+----------------------+-----------+---------------+---------------+---+---+---+---+---+
|  1|    M14860|   M|              298.1|                  308.6|                  1551|       42.8|              0|              0|  0|  0|  0|  0|  0|
|  2|    L47181|   L|              298.2|                  308.7|                  1408|       46.3|              3|              0|  0|  0|  0|  0|  0|
|  3|    L47182|   L|              298.1|                  308.5|                  1498|       49.4|              5|              0|  0|  0|  0|  0|  0|
|  4|    L47183|   L|              298.2|                  308.6|                 

In [ ]:
print("Rows :", df.count())
print("Columns :", len(df.columns))

Rows : 10000
Columns : 14


In [ ]:
#Transform
clean_df = df.drop("UDI", "Product ID")
clean_df.show(5)

+----+-------------------+-----------------------+----------------------+-----------+---------------+---------------+---+---+---+---+---+
|Type|Air temperature [K]|Process temperature [K]|Rotational speed [rpm]|Torque [Nm]|Tool wear [min]|Machine failure|TWF|HDF|PWF|OSF|RNF|
+----+-------------------+-----------------------+----------------------+-----------+---------------+---------------+---+---+---+---+---+
|   M|              298.1|                  308.6|                  1551|       42.8|              0|              0|  0|  0|  0|  0|  0|
|   L|              298.2|                  308.7|                  1408|       46.3|              3|              0|  0|  0|  0|  0|  0|
|   L|              298.1|                  308.5|                  1498|       49.4|              5|              0|  0|  0|  0|  0|  0|
|   L|              298.2|                  308.6|                  1433|       39.5|              7|              0|  0|  0|  0|  0|  0|
|   L|              298.2|        

In [ ]:
df.na.drop().show(5)

+---+----------+----+-------------------+-----------------------+----------------------+-----------+---------------+---------------+---+---+---+---+---+
|UDI|Product ID|Type|Air temperature [K]|Process temperature [K]|Rotational speed [rpm]|Torque [Nm]|Tool wear [min]|Machine failure|TWF|HDF|PWF|OSF|RNF|
+---+----------+----+-------------------+-----------------------+----------------------+-----------+---------------+---------------+---+---+---+---+---+
|  1|    M14860|   M|              298.1|                  308.6|                  1551|       42.8|              0|              0|  0|  0|  0|  0|  0|
|  2|    L47181|   L|              298.2|                  308.7|                  1408|       46.3|              3|              0|  0|  0|  0|  0|  0|
|  3|    L47182|   L|              298.1|                  308.5|                  1498|       49.4|              5|              0|  0|  0|  0|  0|  0|
|  4|    L47183|   L|              298.2|                  308.6|                 

In [ ]:
print("Rows After Cleaning :", clean_df.count())

Rows After Cleaning : 10000


In [ ]:
clean_df.write.mode("overwrite").csv(
    "/content/cleaned_data",
    header=True
)

In [ ]:
print("ETL Pipeline Completed Successfully")

ETL Pipeline Completed Successfully


#Machine Learning/Deep Learning Implementation

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [ ]:
spark = SparkSession.builder.appName("Predictive Maintenance").getOrCreate()

In [ ]:
df = spark.read.csv(
    "/content/cleaned_data",
    header=True,
    inferSchema=True
)

In [ ]:
indexer = StringIndexer(
    inputCol="Type",
    outputCol="TypeIndex"
)

df = indexer.fit(df).transform(df)

In [ ]:
assembler = VectorAssembler(
inputCols=[
"Air temperature [K]",
"Process temperature [K]",
"Rotational speed [rpm]",
"Torque [Nm]",
"Tool wear [min]",
"TypeIndex"
],

outputCol="features"

)

data = assembler.transform(df)

In [ ]:
final_data = data.select("features","Machine failure")
final_data = final_data.withColumnRenamed(
    "Machine failure",
    "label"
)

final_data.show(5)

+--------------------+-----+
|            features|label|
+--------------------+-----+
|[298.1,308.6,1551...|    0|
|[298.2,308.7,1408...|    0|
|[298.1,308.5,1498...|    0|
|[298.2,308.6,1433...|    0|
|[298.2,308.7,1408...|    0|
+--------------------+-----+
only showing top 5 rows


In [ ]:
train,test = final_data.randomSplit([0.8,0.2], seed=10)

In [ ]:
lr = LogisticRegression()
model = lr.fit(train)

In [ ]:
prediction = model.transform(test)

prediction.select(
    "label",
    "prediction"
).show(10)

+-----+----------+
|label|prediction|
+-----+----------+
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
+-----+----------+
only showing top 10 rows


In [ ]:
prediction.groupBy("label").count().show()

+-----+-----+
|label|count|
+-----+-----+
|    1|   62|
|    0| 1927|
+-----+-----+



In [ ]:
prediction.groupBy("prediction").count().show()

+----------+-----+
|prediction|count|
+----------+-----+
|       0.0| 1969|
|       1.0|   20|
+----------+-----+



In [ ]:
prediction.filter(prediction["label"] == 1).select(
    "label",
    "prediction"
).show(20)

+-----+----------+
|label|prediction|
+-----+----------+
|    1|       0.0|
|    1|       1.0|
|    1|       0.0|
|    1|       0.0|
|    1|       0.0|
|    1|       0.0|
|    1|       0.0|
|    1|       0.0|
|    1|       0.0|
|    1|       0.0|
|    1|       1.0|
|    1|       1.0|
|    1|       0.0|
|    1|       1.0|
|    1|       1.0|
|    1|       0.0|
|    1|       0.0|
|    1|       1.0|
|    1|       0.0|
|    1|       0.0|
+-----+----------+
only showing top 20 rows


In [ ]:
evaluator = MulticlassClassificationEvaluator(
labelCol="label",
predictionCol="prediction",
metricName="accuracy"
)


In [ ]:
accuracy = evaluator.evaluate(prediction)
print("Accuracy =", accuracy)

Accuracy = 0.975867269984917


In [ ]:
df.groupBy("Machine failure").count().show()

+---------------+-----+
|Machine failure|count|
+---------------+-----+
|              1|  339|
|              0| 9661|
+---------------+-----+



In [ ]:
prediction.groupBy("label", "prediction").count().show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    1|       0.0|   45|
|    0|       0.0| 1924|
|    1|       1.0|   17|
|    0|       1.0|    3|
+-----+----------+-----+

